# Trading AI Fine-Tune Setup
- Model: DeepSeek-Coder-6.7b-Instruct (text-focused).
- Flow: Form for data → Load/tokenize → LoRA train → Test.
- Goal: Learn your patterns (stoch crosses, triangles) + screening/deriv flags.

In [ ]:
import os
import warnings

# 1. Kill all loading widgets globally to prevent visual glitches
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
# 2. Mute harmless Hugging Face warnings
warnings.filterwarnings("ignore")

!pip install -q ipywidgets transformers accelerate datasets peft trl
!pip install -U -q bitsandbytes>=0.46.1

# 3. Connect to your 2TB Google Drive!
from google.colab import drive
try:
    drive.mount('/content/drive')
    print("✅ Google Drive connected!")
except:
    print("Google Drive not connected.")

import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling

# Storage (Make sure you uploaded your_trades.jsonl to your Google Drive!)
jsonl_path = "/content/drive/MyDrive/your_trades.jsonl"
# Change this to whatever 15GB model you decided to use!
model_name = "deepseek-ai/deepseek-coder-6.7b-instruct"

print("Setup done. GPU is Active:", torch.cuda.is_available())

In [ ]:
import os
import json

# FIXED: We need to tell Python that this is an empty list first!
dataset_entries = []

if os.path.exists(jsonl_path):
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        # Now it knows where to put your data
        dataset_entries[:] = [json.loads(line) for line in f]
    print(f"✅ Auto-loaded {len(dataset_entries)} saved trades – everything is back!")
else:
    print(f"⚠️ Could not find the file at {jsonl_path}. Please double check the path!")

In [ ]:
# FIXED CELL 3 – never lose data again
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import os

prompt_text = widgets.Textarea(placeholder="E.g., Analyze chart: BTC from Bybit screen...", description='Prompt:', rows=3, layout=widgets.Layout(width='80%'))
completion_text = widgets.Textarea(placeholder="E.g., Entry: 80000, TP: ..., SL: ..., Clarify: ...", description='Completion:', rows=3, layout=widgets.Layout(width='80%'))
submit_btn = widgets.Button(description="Add Entry", button_style='success')
status_label = widgets.Label(value="Ready to add.")
output = widgets.Output()

def validate_entry(prompt, completion):
    required = ['entry', 'sl', 'tp']
    completion_lower = completion.lower()
    missing = [r for r in required if r not in completion_lower]
    if missing:
        return f"Missing basics: {', '.join([r.upper() for r in missing])}. Clarify?"
    return None

@output.capture()
def on_submit(b):
    p, c = prompt_text.value.strip(), completion_text.value.strip()
    if not p or not c:
        status_label.value = "Error: Both fields required."
        return
    issue = validate_entry(p, c)
    if issue:
        status_label.value = issue
        return

    # ←←← THIS PART IS NOW BULLET-PROOF ←←←
    entry = {"prompt": p, "completion": c}
    dataset_entries.append(entry)
    with open(jsonl_path, 'a', encoding='utf-8') as f:   # ← append mode = never overwrite
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    # reload from disk so count is always correct
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        dataset_entries[:] = [json.loads(line) for line in f]
    # ←←← END OF BULLET-PROOF PART ←←←

    status_label.value = f"Added! Total: {len(dataset_entries)}"
    prompt_text.value = ''; completion_text.value = ''

submit_btn.on_click(on_submit)
form = widgets.VBox([prompt_text, completion_text, submit_btn, status_label])
display(form, output)
print("Form ready – your data is now 100% safe even if you close Jupyter Lab.")

In [ ]:
## Load & Prepare Data Run this after adding trades.

from datasets import load_dataset

print("Loading dataset...")

dataset = load_dataset("json", data_files=jsonl_path, split="train")

if len(dataset) == 0:
    print("❌ No data found! Add more trades in the form.")
else:
    dataset = dataset.train_test_split(test_size=0.2)
    print(f"✅ Loaded {len(dataset['train'])} training examples and {len(dataset['test'])} test examples")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def format_ex(example):
    text = f"{example['prompt']} {example['completion']}"
    return tokenizer(text, truncation=True, padding="max_length", max_length=512, return_tensors="pt")

tokenized_dataset = dataset.map(lambda ex: {
    "input_ids": format_ex({"prompt": ex["prompt"], "completion": ex["completion"]})["input_ids"].squeeze(),
    "attention_mask": format_ex({"prompt": ex["prompt"], "completion": ex["completion"]})["attention_mask"].squeeze()
}, remove_columns=dataset["train"].column_names)

print("✅ Data tokenized and ready!")
if len(tokenized_dataset['train']) > 0:
    print(f"Tokens per example: {len(tokenized_dataset['train'][0]['input_ids'])}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "microsoft/Phi-3-mini-4k-instruct"

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Configuring 4-bit GPU Compression...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print(f"Downloading and compressing {model_name} directly to the GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Deep-Scrubbing hidden BFloat16 Parameters AND Buffers...")
# 1. Scrub all normal parameters
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

# 2. Scrub the hidden buffers (This is what crashed your code!)
for buffer in model.buffers():
    if buffer.dtype == torch.bfloat16:
        buffer.data = buffer.data.to(torch.float16)

model = prepare_model_for_kbit_training(model)

print("Applying LoRA Configuration...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# 3. Double-check the LoRA adapters to ensure strict stability
for param in model.parameters():
    if param.requires_grad and param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)

model.print_trainable_parameters()
print("✅ PHI-3 MODEL completely sanitized! No hiding places left.")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling

print("Preparing trainer...")

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./trading_lora_model",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    optim="adamw_torch",
    warmup_steps=2,
    logging_steps=1,
    eval_strategy="epoch" if len(tokenized_dataset["test"]) > 0 else "no",
    save_strategy="epoch",
    fp16=False,                         # FIXED: Turned OFF to bypass the GradScaler crash!
    bf16=False,
    gradient_checkpointing=True,
    report_to="none",
    dataloader_pin_memory=False,
    remove_unused_columns=False
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"] if len(tokenized_dataset["test"]) > 0 else None,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("🚀 Starting training... (Bypassing the GradScaler!)")

trainer.train()

trainer.save_model("./trading_lora_model")
tokenizer.save_pretrained("./trading_lora_model")

print("✅ Training finished and model saved safely to your Drive!")

In [ ]:
print("Testing your newly trained AI...")

# Test with your style
test_prompt = "Analyze chart: TESTCOIN from Bybit screen: Exotic alt, stochastic %K cross above %D on daily (5,3,3), K value < 20 before crossing up. Timeframe: 4H. Visual pattern: Ascending triangle with resistance at 0.05 (touched three times). Structure: HH/HL intact. Strategy match for long entry?"

# Format for Phi-3's brain
formatted_prompt = f"<|user|>\n{test_prompt}<|end|>\n<|assistant|>\n"

# The model and tokenizer are already in memory! Just use them directly:
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n📈 AI Response:")
print(response.split("<|assistant|>")[-1].strip())

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

print("=== 📈 Daily AI Trading Assistant Ready ===")

# 1. Create the Text Box
prompt_input = widgets.Textarea(
    value="Analyze chart: TESTCOIN from Bybit screen: Exotic alt, stochastic %K cross above %D on daily (5,3,3), K value < 20 before crossing up. Timeframe: 4H. Visual pattern: Ascending triangle with resistance at 0.05 (touched three times). Structure: HH/HL intact. Strategy match for long entry?",
    placeholder="Type your Bybit chart setup here...",
    description="Setup:",
    layout=widgets.Layout(width="90%", height="100px")
)

# 2. Create the Clickable Button
analyze_button = widgets.Button(
    description="Analyze Trade",
    button_style="success", # Gives you a nice green "Go" button
    icon="check"
)

# 3. Create an output area so it doesn't spam your notebook screen
output_area = widgets.Output()

# 4. Define what the button does when clicked
def on_analyze_click(b):
    with output_area:
        clear_output() # Clears your previous trade analysis
        print("⏳ Processing chart setup... please wait...\n")

        user_text = prompt_input.value

        # Format strictly for Phi-3 so it knows we are chatting
        formatted_prompt = f"<|user|>\n{user_text}<|end|>\n<|assistant|>\n"

        # Load the input into the GPU
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

        # The AI Brain (With all anti-hallucination protections turned ON)
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,        # Cold, strict logic
            do_sample=True,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,   # The Stop Sign
            pad_token_id=tokenizer.pad_token_id
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        final_answer = response.split("<|assistant|>")[-1].strip()

        print("=== 🤖 AI Response ===")
        print(final_answer)
        print("\n" + "-"*50) # Just a visual divider

# 5. Connect the click action to the button
analyze_button.on_click(on_analyze_click)

# 6. Display the beautiful new UI
display(prompt_input, analyze_button, output_area)

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Loading saved model from Google Drive...")

model_name = "microsoft/Phi-3-mini-4k-instruct"
lora_path = "./trading_lora_model" # Make sure this points to your Google Drive path if you saved it there!

tokenizer = AutoTokenizer.from_pretrained(lora_path)

# 1. We MUST use 4-bit compression again, or the T4 GPU will crash
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 2. FIXED: device_map="auto" (Use GPU, not CPU!)
# 3. FIXED: Removed trust_remote_code=True
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

# Merge your custom trading brain (LoRA) into the base model
model = PeftModel.from_pretrained(base_model, lora_path)
print("Model loaded successfully!")

test_prompt = "Analyze chart: TESTCOIN from Bybit screen: Exotic alt, stochastic %K cross above %D on daily (5,3,3), K value < 20 before crossing up. Timeframe: 4H. Visual pattern: Ascending triangle with resistance at 0.05 (touched three times). Structure: HH/HL intact. Strategy match for long entry?"
formatted_prompt = f"<|user|>\n{test_prompt}<|end|>\n<|assistant|>\n"

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n📈 AI Response:")
print(response.split("<|assistant|>")[-1].strip())